<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/18-computer-vision/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 18: Exercises

Twelve exercises: four Easy, five Medium and three Hard. The Easy ones are the small calculations you'll do all the time when working with images: output sizes, converting images to tensors, counting a layer's parameters, and turning a network's scores into readable predictions. The Medium ones build the parts of a CNN and its training by hand: a convolution and max pooling in NumPy, the receptive field of a stack of layers, the shift used for data augmentation, and a pretrained ResNet prepared for transfer learning. The Hard ones go all the way: a fast batched convolution with several channels, ResNet's residual block, and the backward pass of a convolution, each checked against PyTorch.

Most checks compare your code with PyTorch or torchvision themselves. Everything runs in a few seconds on a CPU, and nothing is downloaded: the ResNet in Exercise 9 is built with random weights.

**Run the setup cell first.**

Rules:

- NumPy, PyTorch and torchvision. Don't call the PyTorch version of what you're writing (such as `F.conv2d` or `F.max_pool2d`); the checks use it to test your answer.
- Solutions are in `03_solutions.ipynb`.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms import v2


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):
        if not (isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor)):
            return False
        a, b = a.detach().cpu().numpy(), b.detach().cpu().numpy()
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol, equal_nan=True))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k], tol) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, torch.Tensor):
        body = np.array2string(x.detach().cpu().numpy(), precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"tensor(shape={tuple(x.shape)}, dtype={str(x.dtype).replace('torch.', '')}, {' '.join(body.split())})"
    elif isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, (list, tuple)) and any(isinstance(i, (np.ndarray, torch.Tensor)) for i in x):
        brackets = "()" if isinstance(x, tuple) else "[]"
        return brackets[0] + ",\n           ".join(_short(i, limit) for i in x) + brackets[1]
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_conv_output_size(f):
    def actual(n, k, p, s, d):
        return F.conv2d(torch.zeros(1, 1, n, n), torch.zeros(1, 1, k, k), padding=p, stride=s, dilation=d).shape[-1]

    cases = [(28, 3, 0, 1, 1), (32, 5, 2, 2, 1), (7, 2, 0, 2, 1), (224, 7, 3, 2, 1), (15, 3, 2, 1, 2), (64, 3, 0, 3, 3)]
    _scenarios("conv_output_size", [
        ("padding 1 keeps the size", lambda: f(28, 3, padding=1), 28, None),
        ("stride 2 halves it", lambda: f(28, 3, padding=1, stride=2), 14, None),
        ("rounds down when the stride doesn't divide evenly", lambda: f(7, 2, stride=2), 3, None),
        ("the same as PyTorch on six layers", lambda: [f(n, k, padding=p, stride=s, dilation=d) for n, k, p, s, d in cases],
         [actual(*c) for c in cases], "with dilation, the kernel covers dilation * (k - 1) + 1 pixels"),
        ("returns an int", lambda: type(f(28, 3)).__name__, "int", "use integer division //"),
        ("a kernel that doesn't fit raises", lambda: f(4, 5), Raises(ValueError), None),
    ])


def check_image_to_tensor(f):
    rng = np.random.default_rng(0)
    colour = rng.integers(0, 256, size=(4, 5, 3), dtype=np.uint8)
    grey = rng.integers(0, 256, size=(6, 7), dtype=np.uint8)
    original = colour.copy()
    expected = torch.tensor(colour, dtype=torch.float32).permute(2, 0, 1) / 255

    def unchanged():
        f(colour)
        return colour

    _scenarios("image_to_tensor", [
        ("colour image shape", lambda: tuple(f(colour).shape), (3, 4, 5), None),
        ("channels in the right place", lambda: f(colour), expected, "use permute, not reshape"),
        ("greyscale gets one channel", lambda: tuple(f(grey).shape), (1, 6, 7), None),
        ("greyscale values", lambda: f(grey)[0], torch.tensor(grey, dtype=torch.float32) / 255, None),
        ("float32", lambda: str(f(colour).dtype), "torch.float32", None),
        ("255 becomes 1.0", lambda: f(np.full((2, 2, 3), 255, dtype=np.uint8)).max().item(), 1.0, None),
        ("the input is not changed", unchanged, original, None),
        ("floats are refused", lambda: f(colour / 255), Raises(ValueError), None),
    ])


def check_conv_parameter_count(f):
    def torch_count(i, o, k, bias=True):
        return sum(p.numel() for p in nn.Conv2d(i, o, k, bias=bias).parameters())

    _scenarios("conv_parameter_count", [
        ("the example", lambda: f(3, 16, 3), 448, "out x in x k x k weights, plus out biases"),
        ("a bigger layer", lambda: f(64, 128, 3), torch_count(64, 128, 3), None),
        ("without bias", lambda: f(64, 128, 3, bias=False), torch_count(64, 128, 3, bias=False), None),
        ("a rectangular kernel", lambda: f(8, 4, (1, 5)), torch_count(8, 4, (1, 5)), "kernel_size can be a pair"),
        ("a 1 x 1 convolution", lambda: f(256, 64, 1, bias=False), torch_count(256, 64, 1, bias=False), None),
    ])


def check_top_k_labels(f):
    names = ["cat", "dog", "fox", "owl", "bee"]
    logits = torch.tensor([0.5, 2.0, -1.0, 2.5, 0.0])
    p = torch.softmax(logits, dim=0)

    def probs(result):
        return [prob for _, prob in result]

    _scenarios("top_k_labels", [
        ("the example", lambda: f(torch.tensor([1.0, 3.0, 2.0]), ["cat", "dog", "fox"], k=2),
         [("dog", torch.softmax(torch.tensor([1.0, 3.0, 2.0]), 0)[1].item()), ("fox", torch.softmax(torch.tensor([1.0, 3.0, 2.0]), 0)[2].item())], None),
        ("names in order, most likely first", lambda: [name for name, _ in f(logits, names, k=3)], ["owl", "dog", "cat"], None),
        ("probabilities use all the classes", lambda: probs(f(logits, names, k=2)), [p[3].item(), p[1].item()],
         "softmax over every class, then pick the top k"),
        ("probabilities are Python floats", lambda: [type(x).__name__ for x in probs(f(logits, names, k=2))], ["float", "float"], "use .tolist() or .item()"),
        ("k = 1", lambda: len(f(logits, names, k=1)), 1, None),
        ("k too large raises", lambda: f(logits, names, k=6), Raises(ValueError), None),
    ])


def check_conv2d_single(f):
    rng = np.random.default_rng(1)
    image = rng.normal(size=(9, 11))
    asym = np.array([[1.0, 2.0, 0.0], [0.0, -1.0, 3.0], [-2.0, 0.5, 1.0]])
    wide = rng.normal(size=(2, 3))

    def torch_conv(img, k, padding=0, stride=1):
        out = F.conv2d(torch.tensor(img).reshape(1, 1, *img.shape), torch.tensor(k).reshape(1, 1, *k.shape),
                       padding=padding, stride=stride)
        return out[0, 0].numpy()

    _scenarios("conv2d_single", [
        ("the lesson's tiny edge example", lambda: f(np.array([[0, 0, 0, 9, 9]] * 4, dtype=float), np.array([[-1.0, 0, 1]] * 3)),
         np.array([[0.0, 27, 27], [0, 27, 27]]), None),
        ("an asymmetric kernel", lambda: f(image, asym), torch_conv(image, asym), "don't flip the kernel"),
        ("with padding", lambda: f(image, asym, padding=1), torch_conv(image, asym, padding=1), None),
        ("with stride 2", lambda: f(image, asym, stride=2), torch_conv(image, asym, stride=2), None),
        ("a rectangular kernel, padding and stride", lambda: f(image, wide, padding=2, stride=3), torch_conv(image, wide, padding=2, stride=3),
         "kernel height and width can differ"),
        ("a 1 x 4 kernel", lambda: f(image, wide[:1, :] @ np.ones((3, 4))), torch_conv(image, wide[:1, :] @ np.ones((3, 4))), None),
        ("returns float64", lambda: str(f(np.ones((4, 4), dtype=np.uint8), np.ones((2, 2))).dtype), "float64", None),
    ])


def check_max_pool2d(f):
    rng = np.random.default_rng(2)
    fm = rng.normal(size=(7, 9))
    ints = rng.integers(0, 100, size=(6, 6))

    def torch_pool(a, size, stride=None):
        return F.max_pool2d(torch.tensor(a, dtype=torch.float64).reshape(1, 1, *a.shape), size, stride)[0, 0].numpy()

    _scenarios("max_pool2d", [
        ("the lesson's example", lambda: f(np.array([[1., 3, 2, 0], [4, 2, 1, 1], [0, 1, 5, 6], [2, 2, 7, 3]])),
         np.array([[4.0, 2], [2, 7]]), None),
        ("an odd size drops the edge", lambda: f(fm), torch_pool(fm, 2), "windows that hang over the edge are dropped"),
        ("size 3", lambda: f(fm, size=3), torch_pool(fm, 3), None),
        ("overlapping windows: size 3, stride 2", lambda: f(fm, size=3, stride=2), torch_pool(fm, 3, 2), "stride defaults to size only when it's None"),
        ("keeps an integer dtype", lambda: str(f(ints).dtype), str(ints.dtype), None),
    ])


def _measured_receptive_field(layers):
    torch.manual_seed(0)
    modules = [nn.Conv2d(1, 1, k, stride=s, bias=False) for k, s in layers]
    for m in modules:
        nn.init.constant_(m.weight, 1.0)
    size = 1
    for k, s in reversed(layers):
        size = (size - 1) * s + k
    size += 4
    x = torch.zeros(1, 1, size, size, requires_grad=True)
    out = x
    for m in modules:
        out = m(out)
    out[0, 0, 0, 0].backward()
    touched = torch.nonzero(x.grad[0, 0].abs().sum(dim=1) > 0).flatten()
    return int(touched.max() - touched.min() + 1)


def check_receptive_field(f):
    rng = np.random.default_rng(3)
    stacks = [[(int(rng.choice([1, 2, 3, 5, 7])), int(rng.choice([1, 1, 2, 3]))) for _ in range(int(rng.integers(2, 6)))] for _ in range(6)]
    _scenarios("receptive_field", [
        ("two 3 x 3 convolutions", lambda: f([(3, 1), (3, 1)]), 5, None),
        ("with a pooling between them", lambda: f([(3, 1), (2, 2), (3, 1)]), 8, "multiply the jump by each stride"),
        ("ResNet's stem", lambda: f([(7, 2), (3, 2)]), 11, None),
        ("the input alone", lambda: f([]), 1, None),
        ("six random stacks, measured with autograd", lambda: [f(s) for s in stacks], [_measured_receptive_field(s) for s in stacks],
         "grow the field by (kernel_size - 1) * jump before updating the jump"),
    ])


def check_pad_and_crop(f):
    g = torch.Generator().manual_seed(4)
    image = torch.rand(3, 6, 8, generator=g)
    positions = [(0, 0), (4, 1), (2, 2), (1, 4), (4, 4)]

    def torchvision_crop(top, left, padding=2):
        return v2.functional.crop(image, top - padding, left - padding, 6, 8)

    _scenarios("pad_and_crop", [
        ("the middle position returns the image", lambda: f(image, 2, 2, 2), image, None),
        ("top = 0, left = 0 moves it down and right", lambda: f(image, 2, 0, 0), torchvision_crop(0, 0), None),
        ("five positions match torchvision", lambda: [f(image, 2, t, l) for t, l in positions],
         [torchvision_crop(t, l) for t, l in positions], "top is a row of the padded image, left a column"),
        ("keeps the shape", lambda: tuple(f(image, 3, 5, 1).shape), (3, 6, 8), None),
        ("a position outside the padding raises", lambda: f(image, 2, 5, 0), Raises(ValueError), None),
    ])


def check_make_transfer_model(f):
    import torchvision

    def build(n_classes, train_last_stage=False):
        torch.manual_seed(0)
        return f(torchvision.models.resnet18(weights=None), n_classes, train_last_stage=train_last_stage)

    def trainable(m):
        return sum(p.numel() for p in m.parameters() if p.requires_grad)

    def trainable_names(m):
        return sorted({name.split(".")[0] for name, p in m.named_parameters() if p.requires_grad})

    _scenarios("make_transfer_model", [
        ("the new last layer", lambda: (build(6).fc.in_features, build(6).fc.out_features), (512, 6), None),
        ("only the new layer is trainable", lambda: trainable(build(6)), 3078, "freeze first, then create the new layer"),
        ("which parts train", lambda: trainable_names(build(6)), ["fc"], None),
        ("with the last stage", lambda: trainable(build(6, train_last_stage=True)), 8396806, None),
        ("which parts train, with the last stage", lambda: trainable_names(build(2, train_last_stage=True)), ["fc", "layer4"], None),
        ("returns the model", lambda: type(build(3)).__name__, "ResNet", None),
        ("works for ResNet-50 too", lambda: f(torchvision.models.resnet50(weights=None), 4).fc.in_features, 2048,
         "read the number of inputs from model.fc.in_features"),
        ("the output has n_classes columns", lambda: tuple(build(3).eval()(torch.zeros(2, 3, 64, 64)).shape), (2, 3), None),
    ])


def check_conv2d_batch(f):
    g = torch.Generator().manual_seed(5)
    x = torch.randn(2, 3, 9, 10, generator=g, dtype=torch.float64)
    w = torch.randn(4, 3, 3, 3, generator=g, dtype=torch.float64)
    b = torch.randn(4, generator=g, dtype=torch.float64)
    w_rect = torch.randn(5, 3, 2, 4, generator=g, dtype=torch.float64)

    def grads(conv):
        xs, ws, bs = (t.clone().requires_grad_(True) for t in (x, w, b))
        (conv(xs, ws, bs, stride=2, padding=1) * torch.arange(4 * 5 * 5, dtype=torch.float64).reshape(4, 5, 5)).sum().backward()
        return [xs.grad, ws.grad, bs.grad]

    def torch_conv(xx, ww, bb=None, stride=1, padding=0):
        return F.conv2d(xx, ww, bb, stride=stride, padding=padding)

    _scenarios("conv2d_batch", [
        ("no padding, stride 1", lambda: f(x, w), F.conv2d(x, w), None),
        ("with bias", lambda: f(x, w, b), F.conv2d(x, w, b), "add the bias to every position of its channel"),
        ("padding 1", lambda: f(x, w, b, padding=1), F.conv2d(x, w, b, padding=1), None),
        ("stride 2 on an odd size", lambda: f(x, w, b, stride=2, padding=1), F.conv2d(x, w, b, stride=2, padding=1),
         "each slice must have exactly H_out rows and W_out columns"),
        ("a rectangular kernel, stride 3", lambda: f(x, w_rect, stride=3, padding=2), F.conv2d(x, w_rect, stride=3, padding=2), None),
        ("gradients match PyTorch's", lambda: grads(f), grads(torch_conv), None),
    ])


def _torchvision_block(c_in, c_out, stride):
    from torchvision.models.resnet import BasicBlock
    torch.manual_seed(0)
    downsample = None
    if stride != 1 or c_in != c_out:
        downsample = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False), nn.BatchNorm2d(c_out))
    block = BasicBlock(c_in, c_out, stride=stride, downsample=downsample)
    with torch.no_grad():
        for m in block.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.weight.uniform_(0.5, 1.5)
                m.bias.uniform_(-0.5, 0.5)
                m.running_mean.uniform_(-0.2, 0.2)
                m.running_var.uniform_(0.5, 1.5)
    return block


def _copy_into(cls, c_in, c_out, stride):
    reference = _torchvision_block(c_in, c_out, stride)
    mine = cls(c_in, c_out, stride)
    state = {k.replace("downsample", "shortcut"): v for k, v in reference.state_dict().items()}
    mine.load_state_dict(state)
    return reference, mine


def check_ResidualBlock(cls):
    x = torch.randn(2, 16, 12, 12, generator=torch.Generator().manual_seed(6))

    def n_params(*args):
        return sum(p.numel() for p in cls(*args).parameters())

    def yours(c_in, c_out, stride, train_mode=False):
        _, mine = _copy_into(cls, c_in, c_out, stride)
        mine.train(train_mode)
        with torch.no_grad():
            return mine(x[:, :c_in])

    def reference(c_in, c_out, stride, train_mode=False):
        block = _torchvision_block(c_in, c_out, stride)
        block.train(train_mode)
        with torch.no_grad():
            return block(x[:, :c_in])

    def shortcut_kind(*args):
        return [type(m).__name__ for m in cls(*args).shortcut]

    _scenarios("ResidualBlock", [
        ("same shape: output shape", lambda: tuple(cls(16, 16)(x).shape), (2, 16, 12, 12), None),
        ("downsampling: output shape", lambda: tuple(cls(16, 32, 2)(x).shape), (2, 32, 6, 6), None),
        ("the identity shortcut has no layers", lambda: shortcut_kind(16, 16), [], "use an empty nn.Sequential()"),
        ("the projection shortcut", lambda: shortcut_kind(16, 32, 2), ["Conv2d", "BatchNorm2d"], None),
        ("same channels with stride 2 needs a projection too", lambda: tuple(cls(16, 16, 2)(x).shape), (2, 16, 6, 6), None),
        ("parameters, same shape", lambda: n_params(16, 16), sum(p.numel() for p in _torchvision_block(16, 16, 1).parameters()), "no bias in the convolutions"),
        ("parameters, downsampling", lambda: n_params(16, 32, 2), sum(p.numel() for p in _torchvision_block(16, 32, 2).parameters()), None),
        ("matches torchvision, same shape", lambda: yours(16, 16, 1), reference(16, 16, 1), "the final ReLU comes after the addition"),
        ("matches torchvision, downsampling", lambda: yours(16, 32, 2), reference(16, 32, 2), "the stride belongs in conv1 and in the shortcut"),
        ("matches torchvision in training mode", lambda: yours(8, 24, 1, True), reference(8, 24, 1, True), None),
    ])


def check_conv2d_backward(f):
    g = torch.Generator().manual_seed(7)

    def case(n, c_in, h, w, c_out, kh, kw, stride, padding):
        x = torch.randn(n, c_in, h, w, generator=g, dtype=torch.float64)
        weight = torch.randn(c_out, c_in, kh, kw, generator=g, dtype=torch.float64)
        out_shape = F.conv2d(x, weight, stride=stride, padding=padding).shape
        grad_out = torch.randn(out_shape, generator=g, dtype=torch.float64)
        xs, ws = x.clone().requires_grad_(True), weight.clone().requires_grad_(True)
        F.conv2d(xs, ws, stride=stride, padding=padding).backward(grad_out)
        return (x, weight, grad_out, stride, padding), (xs.grad, ws.grad)

    simple = case(1, 1, 5, 5, 1, 3, 3, 1, 0)
    channels = case(2, 3, 7, 8, 4, 3, 3, 1, 1)
    strided = case(2, 2, 9, 9, 3, 3, 3, 2, 1)
    gaps = case(1, 2, 8, 8, 2, 2, 2, 3, 0)
    rect = case(2, 3, 6, 9, 2, 2, 4, 2, 2)

    def run(c):
        x, weight, grad_out, stride, padding = c[0]
        return f(x, weight, grad_out, stride=stride, padding=padding)

    _scenarios("conv2d_backward", [
        ("one channel, no padding", lambda: run(simple), simple[1], None),
        ("several channels, padding 1", lambda: run(channels), channels[1], "remove the padding from the input gradient at the end"),
        ("stride 2", lambda: run(strided), strided[1], "add into strided slices, with += where they overlap"),
        ("stride larger than the kernel", lambda: run(gaps), gaps[1], None),
        ("a rectangular kernel", lambda: run(rect), rect[1], None),
        ("shapes", lambda: [tuple(t.shape) for t in run(channels)], [tuple(t.shape) for t in channels[1]], None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `conv_output_size`

Write `conv_output_size(n, kernel_size, padding=0, stride=1, dilation=1)` that returns the output size (height or width) of a convolution or pooling layer applied to an input of size `n`.

- Without dilation, the formula is `(n + 2 * padding - kernel_size) // stride + 1`.
- **Dilation** spreads the kernel out, leaving gaps of `dilation - 1` pixels between its weights, so a kernel of size `k` covers `dilation * (k - 1) + 1` pixels. Use that covered size in place of `kernel_size`.
- Raise a `ValueError` if the kernel doesn't fit at all (the result would be less than 1).

Example: `conv_output_size(28, 3, padding=1)` is `28`, and `conv_output_size(28, 3, padding=1, stride=2)` is `14`.

In [2]:
def conv_output_size(n: int, kernel_size: int, padding: int = 0, stride: int = 1, dilation: int = 1) -> int:
    # your code here
    pass

In [3]:
check_conv_output_size(conv_output_size)

FAIL  conv_output_size: padding 1 keeps the size
      expected: 28
      got:      None


### Exercise 2: `image_to_tensor`

Write `image_to_tensor(image)` that converts an image as loaded by NumPy, Pillow or scikit-image into the form PyTorch expects.

- The input is a NumPy array of `uint8` values from 0 to 255, either colour with shape `(H, W, C)` or greyscale with shape `(H, W)`.
- Return a `float32` tensor of shape `(C, H, W)` with values from 0 to 1. A greyscale image becomes `(1, H, W)`.
- Don't change the input array, and don't share memory with it.
- Raise a `ValueError` if the array isn't `uint8`.

Example: a `(400, 600, 3)` photo becomes a tensor of shape `(3, 400, 600)`, and a pixel value of 255 becomes 1.0.

In [4]:
def image_to_tensor(image: np.ndarray) -> torch.Tensor:
    # your code here
    pass

In [5]:
check_image_to_tensor(image_to_tensor)

FAIL  image_to_tensor: colour image shape
      raised AttributeError: 'NoneType' object has no attribute 'shape'
      expected: (3, 4, 5)


### Exercise 3: `conv_parameter_count`

Write `conv_parameter_count(in_channels, out_channels, kernel_size, bias=True)` that returns the number of parameters in an `nn.Conv2d` layer, without building one.

- `kernel_size` is either one number (a square kernel) or a pair `(height, width)`.
- Each output channel has a kernel for every input channel, plus one bias if `bias` is true.

Example: `conv_parameter_count(3, 16, 3)` is `448` (16 x 3 x 3 x 3 weights plus 16 biases).

In [6]:
def conv_parameter_count(in_channels: int, out_channels: int, kernel_size, bias: bool = True) -> int:
    # your code here
    pass

In [7]:
check_conv_parameter_count(conv_parameter_count)

FAIL  conv_parameter_count: the example
      expected: 448
      got:      None
      hint: out x in x k x k weights, plus out biases


### Exercise 4: `top_k_labels`

Write `top_k_labels(logits, class_names, k=3)` that turns a classifier's output for **one** image into readable predictions.

- `logits` is a 1D tensor with one score per class; `class_names` is a list of the class names in the same order.
- Convert the scores to probabilities with softmax, and return the `k` most likely classes as a list of `(name, probability)` pairs, most likely first. Each probability should be a Python `float`.
- Raise a `ValueError` if `k` is larger than the number of classes.

Example: `top_k_labels(torch.tensor([1.0, 3.0, 2.0]), ["cat", "dog", "fox"], k=2)` is about `[("dog", 0.665), ("fox", 0.245)]`.

In [8]:
def top_k_labels(logits: torch.Tensor, class_names, k: int = 3):
    # your code here
    pass

In [9]:
check_top_k_labels(top_k_labels)

FAIL  top_k_labels: the example
      expected: [('dog', 0.6652408838272095), ('fox', 0.2447284460067749)]
      got:      None


## Medium

### Exercise 5: `conv2d_single`

Write `conv2d_single(image, kernel, padding=0, stride=1)` that applies one kernel to one greyscale image, exactly as `F.conv2d` does (that is, cross-correlation, without flipping the kernel).

- `image` and `kernel` are 2D NumPy arrays. The kernel doesn't have to be square.
- `padding` adds that many rows and columns of zeros on every side first.
- `stride` is the step between positions, in both directions.
- Return a 2D NumPy array of `float64`.

Use NumPy only; the check compares your answer with `F.conv2d`.

In [10]:
def conv2d_single(image: np.ndarray, kernel: np.ndarray, padding: int = 0, stride: int = 1) -> np.ndarray:
    # your code here
    pass

In [11]:
check_conv2d_single(conv2d_single)

FAIL  conv2d_single: the lesson's tiny edge example
      expected: array(shape=(2, 3), dtype=float64, [[ 0., 27., 27.], [ 0., 27., 27.]])
      got:      None


### Exercise 6: `max_pool2d`

Write `max_pool2d(feature_map, size=2, stride=None)` that applies max pooling to a 2D NumPy array, matching `F.max_pool2d`.

- Each output value is the largest value in a `size` x `size` window.
- `stride` is the step between windows; if it's `None`, use `size` (windows side by side, not overlapping).
- Windows that would hang over the edge are dropped, as in PyTorch: a 5 x 5 map pooled with size 2 gives a 2 x 2 result.
- Return a NumPy array with the same dtype as the input.

In [12]:
def max_pool2d(feature_map: np.ndarray, size: int = 2, stride=None) -> np.ndarray:
    # your code here
    pass

In [13]:
check_max_pool2d(max_pool2d)

FAIL  max_pool2d: the lesson's example
      expected: array(shape=(2, 2), dtype=float64, [[4., 2.], [2., 7.]])
      got:      None


### Exercise 7: `receptive_field`

Write `receptive_field(layers)` that returns the receptive field (in pixels, along one side) of a unit in the last layer of a stack of convolution and pooling layers.

- `layers` is a list of `(kernel_size, stride)` pairs, from the input to the output. Padding doesn't affect the receptive field.
- Start with a receptive field of 1 pixel and a **jump** of 1 (how many input pixels apart neighbouring units are). For each layer in order, the receptive field grows by `(kernel_size - 1) * jump`, and then the jump is multiplied by the layer's stride.
- An empty list means the input itself: a receptive field of 1.

Example: two 3 x 3 convolutions with stride 1, `[(3, 1), (3, 1)]`, give 5. Putting a 2 x 2 pooling between them, `[(3, 1), (2, 2), (3, 1)]`, gives 8.

In [14]:
def receptive_field(layers) -> int:
    # your code here
    pass

In [15]:
check_receptive_field(receptive_field)

FAIL  receptive_field: two 3 x 3 convolutions
      expected: 5
      got:      None


### Exercise 8: `pad_and_crop`

Write `pad_and_crop(image, padding, top, left)`, the deterministic core of the random-shift augmentation `v2.RandomCrop(size, padding=...)`.

- `image` is a tensor of shape `(C, H, W)`.
- Pad it with `padding` rows or columns of zeros on every side, giving `(C, H + 2 * padding, W + 2 * padding)`.
- Then cut out an `H` x `W` window whose top-left corner is at row `top`, column `left` of the padded image, and return it.
- `top` and `left` must be between 0 and `2 * padding`; otherwise raise a `ValueError`.

With `top = left = padding` the image comes back unchanged. Smaller values move the content down or right, larger values move it up or left. Random augmentation simply picks `top` and `left` at random for every image.

In [16]:
def pad_and_crop(image: torch.Tensor, padding: int, top: int, left: int) -> torch.Tensor:
    # your code here
    pass

In [17]:
check_pad_and_crop(pad_and_crop)

FAIL  pad_and_crop: the middle position returns the image
      expected: tensor(shape=(3, 6, 8), dtype=float32, [[[5.5964e-01, 5.5909e-01, 9.1457e-02, ..., 3.89...
      got:      None


### Exercise 9: `make_transfer_model`

Write `make_transfer_model(model, n_classes, train_last_stage=False)` that prepares a torchvision ResNet for transfer learning.

- Freeze every parameter of `model` (`requires_grad = False`).
- Replace `model.fc` with a new `nn.Linear` that has the same number of inputs as the old one and `n_classes` outputs. The new layer's parameters must be trainable.
- If `train_last_stage` is true, also make the parameters of `model.layer4` trainable again.
- Return the model.

The check uses `torchvision.models.resnet18(weights=None)` (random weights, so nothing needs downloading), but your function should work for any ResNet.

In [18]:
def make_transfer_model(model: nn.Module, n_classes: int, train_last_stage: bool = False) -> nn.Module:
    # your code here
    pass

In [19]:
check_make_transfer_model(make_transfer_model)

FAIL  make_transfer_model: the new last layer
      raised AttributeError: 'NoneType' object has no attribute 'fc'
      expected: (512, 6)


## Hard

### Exercise 10: `conv2d_batch`

Write `conv2d_batch(x, weight, bias=None, stride=1, padding=0)` that does what `F.conv2d` does for whole batches, with several input and output channels, using only basic tensor operations.

- `x` has shape `(N, C_in, H, W)`; `weight` has shape `(C_out, C_in, kH, kW)`; `bias`, if given, has shape `(C_out,)`.
- Return a tensor of shape `(N, C_out, H_out, W_out)`.
- Gradients must flow back to `x`, `weight` and `bias` (use ordinary tensor operations; no NumPy, no in-place changes to the inputs).
- Don't use `F.conv2d`, `nn.Conv2d`, `F.unfold` or `nn.Unfold`. You may use `F.pad`.

Hint: instead of looping over output positions, loop over the `kH * kW` positions **inside the kernel**. For kernel position `(i, j)`, the pixels it touches form a strided slice of the padded input, of shape `(N, C_in, H_out, W_out)`, and multiplying by `weight[:, :, i, j]` and summing over input channels is a single `torch.einsum`.

In [20]:
def conv2d_batch(x: torch.Tensor, weight: torch.Tensor, bias=None, stride: int = 1, padding: int = 0) -> torch.Tensor:
    # your code here
    pass

In [21]:
check_conv2d_batch(conv2d_batch)

FAIL  conv2d_batch: no padding, stride 1


      expected: tensor(shape=(2, 4, 7, 8), dtype=float64, [[[[ -8.9019, 3.852 , -4.4016, ..., -8.0525, ...
      got:      None


### Exercise 11: `ResidualBlock`

Write a class `ResidualBlock(in_channels, out_channels, stride=1)`, the basic block of ResNet-18, as an `nn.Module`.

**Main path**, with exactly these attribute names:

- `self.conv1`: 3 x 3 convolution from `in_channels` to `out_channels`, with the given `stride`, padding 1, no bias
- `self.bn1`: batch normalisation over `out_channels`
- `self.conv2`: 3 x 3 convolution from `out_channels` to `out_channels`, stride 1, padding 1, no bias
- `self.bn2`: batch normalisation over `out_channels`

**Shortcut**, `self.shortcut`:

- If `stride == 1` and `in_channels == out_channels`, an empty `nn.Sequential()` (which returns its input unchanged).
- Otherwise `nn.Sequential(conv, bn)`: a 1 x 1 convolution from `in_channels` to `out_channels` with the given `stride` and no bias, followed by batch normalisation, so that the shortcut's output has the same shape as the main path's.

**Forward**: `relu(bn2(conv2(relu(bn1(conv1(x))))) + shortcut(x))`.

The check copies the weights of torchvision's own ResNet block into yours and compares the outputs.

In [22]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        # your code here

    def forward(self, x):
        # your code here
        pass

In [23]:
check_ResidualBlock(ResidualBlock)

FAIL  ResidualBlock: same shape: output shape
      raised AttributeError: 'NoneType' object has no attribute 'shape'
      expected: (2, 16, 12, 12)


### Exercise 12: `conv2d_backward`

Write `conv2d_backward(x, weight, grad_out, stride=1, padding=0)` that computes the gradients of a convolution **without autograd**: the backward pass that PyTorch runs for you.

- `x` has shape `(N, C_in, H, W)` and `weight` has shape `(C_out, C_in, kH, kW)`, as in `F.conv2d(x, weight, stride=stride, padding=padding)` (no bias).
- `grad_out` has the shape of that convolution's output: the gradient of the loss with respect to each output value.
- Return `(grad_x, grad_weight)`, with the same shapes as `x` and `weight`.
- Don't call `backward()` or anything in `torch.autograd`; the check uses them to test your answer.

Hint: start from the loop in `conv2d_batch`. For kernel position `(i, j)`, the output got `einsum("nchw,oc->nohw", patch, weight[:, :, i, j])`. Each of the two inputs of that product gets its share of the gradient: `weight[:, :, i, j]` gets `grad_out` combined with `patch`, and `patch` gets `grad_out` combined with `weight[:, :, i, j]`, which you add into the right strided slice of a padded gradient for `x`. Finally, remove the padding.

In [24]:
def conv2d_backward(x: torch.Tensor, weight: torch.Tensor, grad_out: torch.Tensor, stride: int = 1, padding: int = 0):
    # your code here
    pass

In [25]:
check_conv2d_backward(conv2d_backward)

FAIL  conv2d_backward: one channel, no padding
      expected: (tensor(shape=(1, 1, 5, 5), dtype=float64, [[[[-0.1477, -0.2633, 0.7705, 0.0812, -0.3372...,
           tensor(shape=(1, 1, 3, 3), dtype=float64, [[[[ 7.5206, -7.6129, 2.8014], [-1.4496, 2.79...)
      got:      None


## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then open the mini project in `04_mini_project.ipynb`, which builds a recycling sorter from a pretrained ResNet.